In [2]:
pip install pandas numpy yfinance statsmodels scikit-learn matplotlib seaborn TA-Lib tqdm

Note: you may need to restart the kernel to use updated packages.


"""
Copyright (c) 2025 Matvei Vasetsov (Матвей Васецов). All rights reserved.
Этот код является интеллектуальной собственностью и защищен авторским правом. Любое использование, копирование или распространение без разрешения автора запрещено.

Общее описание:
Этот код реализует комплексную систему для анализа и бэктестинга торговых стратегий на финансовых рынках, поддерживая акции (например, GBTC, BITF) и криптовалюты (например, BTC, ETH). Система выполняет следующие ключевые функции:
1. **Загрузка данных**: Загружает исторические данные о ценах из локальных CSV-файлов, расположенных в директории 'C:\Users\MV\DataSC_raw' (например, GBTC.csv, BTC.csv), с фильтрацией по начальной дате (2019-10-30).
2. **Предобработка данных**: Очищает данные, удаляя дубликаты, некорректные значения и выбросы с использованием метода межквартильного размаха (IQR), а затем нормализует их для моделей машинного обучения (ML).
3. **Генерация признаков**: Создает технические индикаторы (например, EMA, RSI, MACD) и лагированные признаки, определяя целевую переменную (1 для роста цены, -1 для падения).
4. **Торговая стратегия**: Реализует техническую стратегию на основе индикаторов (покупка: цена > EMA20, RSI < 65, MACD > Signal; продажа: цена < EMA20, RSI > 35, MACD < Signal) и оптимизирует её параметры по коэффициенту Шарпа.
5. **Машинное обучение**: Обучает три ML-модели (градиентный бустинг, случайный лес, логистическая регрессия) для прогнозирования направления цены, оптимизируя гиперпараметры через GridSearchCV с кросс-валидацией по временным рядам.
6. **Бэктестинг**: Оценивает стратегии (техническую и ML) на обучающей (80%) и валидационной (20%) выборках, рассчитывая доходность с учетом транзакционных издержек (0.1%) и метрики производительности (например, Шарпа, максимальная просадка).
7. **Визуализация**: Предоставляет интерактивный дашборд с использованием Plotly и ipywidgets, отображающий графики цен с сигналами покупки/продажи, кумулятивную доходность, коэффициент выигрыша, просадку и метрики.

Код модульный, с классами для обработки данных, моделирования ML, реализации стратегий, бэктестинга и визуализации. Устойчивость обеспечивается обработкой ошибок и логированием, что делает его подходящим для трейдеров и аналитиков для сравнения традиционных и ML-стратегий.
"""

In [20]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import Lasso, Ridge, ElasticNet
import matplotlib.pyplot as plt
import talib as ta
import logging
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itertools import product

#Copyright (c) 2025 Matvei Vasetsov (Матвей Васецов). All rights reserved.
#Этот код является интеллектуальной собственностью и защищен авторским правом. Любое использование, копирование или распространение без разрешения автора запрещено.

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class DataHandler:
    def __init__(self):
        self.scaler = StandardScaler()
        self.local_data_path = r'C:\Users\MV\DataSC_raw'  # Путь к локальным данным
        
    def _load_from_local(self, symbol):
        """Загрузка данных из локального файла <symbol>.csv без суффикса -USD"""
        try:
            # Удаляем суффикс -USD для криптовалют
            file_symbol = symbol.replace('-USD', '')
            file_path = Path(self.local_data_path) / f"{file_symbol}.csv"
            if file_path.exists():
                # Чтение CSV файла с ожидаемыми колонками
                data = pd.read_csv(
                    file_path,
                    usecols=['Date', 'Close'],  # Извлекаем только Date и Close
                    parse_dates=['Date'],
                    index_col='Date'
                )
                # Проверяем, что данные содержат Close и не пусты
                if 'Close' in data.columns and not data.empty:
                    data = data[['Close']].copy()
                    data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
                    data = data.dropna()
                    data = data.sort_index()
                    logging.info(f"Loaded data for {symbol} from local file: {file_path}")
                    return data
                else:
                    logging.warning(f"Local file for {symbol} is empty or missing Close column: {file_path}")
                    return None
            logging.info(f"Local file for {symbol} not found at {file_path}")
            return None
        except Exception as e:
            logging.error(f"Error loading local data for {symbol}: {str(e)}")
            return None
        
    def load_data(self, symbols, start_date):
        """Загрузка данных из локальных файлов"""
        data_dict = {}
        
        # Ensure symbols is a list
        if isinstance(symbols, str):
            symbols = [symbols]
            
        for symbol in symbols:
            try:
                # Validate symbol format
                symbol = str(symbol).strip().upper()
                if not symbol:
                    logging.warning("Empty symbol provided, skipping")
                    continue
                    
                logging.info(f"Processing data for {symbol}")
                
                # Загружаем данные из локального файла
                local_data = self._load_from_local(symbol)
                if local_data is not None and not local_data.empty:
                    # Фильтруем данные по дате
                    local_data = local_data[local_data.index >= pd.to_datetime(start_date)]
                    if not local_data.empty:
                        data_dict[symbol] = local_data
                        logging.info(f"Successfully loaded local data for {symbol}: {local_data.shape[0]} records")
                    else:
                        logging.warning(f"Local data for {symbol} is empty after date filtering")
                else:
                    logging.error(f"Failed to load data for {symbol} from local file")
                
            except Exception as e:
                logging.error(f"Unexpected error processing {symbol}: {str(e)}")
                continue
                
        if not data_dict:
            logging.warning("No data was loaded for any symbol")
            
        return data_dict

    def preprocess_data(self, data):
        """Clean and preprocess the data with progress bar"""
        try:
            if data is None or (isinstance(data, pd.DataFrame) and data.empty):
                raise ValueError("Empty or None data provided")
                
            df = data.copy()
            
            # Create progress bar
            pbar = tqdm(total=5, desc="Preprocessing Data")
            
            # Ensure we have a DataFrame
            if isinstance(df, pd.Series):
                df = df.to_frame(name='Close')
            pbar.update(1)
            
            # Convert to numeric and remove any non-numeric values
            df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
            df = df.dropna()
            pbar.update(1)
            
            # Remove duplicates
            df = df[~df.index.duplicated(keep='first')]
            pbar.update(1)
            
            # Remove outliers using IQR method
            if len(df) > 10:
                Q1 = df['Close'].quantile(0.25)
                Q3 = df['Close'].quantile(0.75)
                IQR = Q3 - Q1
                filter_mask = (df['Close'] >= (Q1 - 3.0 * IQR)) & (df['Close'] <= (Q3 + 3.0 * IQR))
                df = df[filter_mask]
            pbar.update(1)
            
            # Final validation
            if df.empty:
                raise ValueError("No valid numeric data after preprocessing")
            
            if not np.issubdtype(df['Close'].dtype, np.number):
                raise ValueError("Close prices are not numeric")
            
            pbar.update(1)
            pbar.close()
                
            logging.info(f"Successfully preprocessed data: {df.shape[0]} records")
            return df
            
        except Exception as e:
            logging.error(f"Error in preprocess_data: {str(e)}")
            return pd.DataFrame()

    def normalize_data(self, data):
        """Normalize data with progress bar"""
        try:
            if data.empty:
                return data
            
            # Create progress bar
            pbar = tqdm(total=3, desc="Normalizing Data")
            
            # Ensure we have numeric data
            data = data.copy()
            data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
            data = data.dropna()
            pbar.update(1)
            
            if data.empty:
                raise ValueError("No valid numeric data to normalize")
            
            # Apply normalization
            normalized_data = pd.DataFrame(
                self.scaler.fit_transform(data),
                columns=data.columns,
                index=data.index
            )
            pbar.update(1)
            
            logging.info(f"Normalized data shape: {normalized_data.shape}")
            
            pbar.update(1)
            pbar.close()
            
            return normalized_data
            
        except Exception as e:
            logging.error(f"Error in normalize_data: {str(e)}")
            return pd.DataFrame()

    def prepare_ml_features(self, data):
        """Prepare features for ML models with explicit dimension handling"""
        try:
            if data.empty:
                raise ValueError("Empty data provided")
            
            df = data.copy()
            
            if isinstance(df, pd.Series):
                df = df.to_frame()
                df.columns = ['Close']
            
            df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
            close_prices = df['Close'].values
            
            if len(close_prices) < 30:
                raise ValueError(f"Insufficient data points: {len(close_prices)}")
            
            # Calculate technical indicators
            try:
                ema20 = ta.EMA(close_prices, timeperiod=20)
                rsi = ta.RSI(close_prices, timeperiod=14)
                macd, signal, _ = ta.MACD(
                    close_prices, 
                    fastperiod=12, 
                    slowperiod=26, 
                    signalperiod=9
                )
                
                df['EMA20'] = pd.Series(ema20, index=df.index).ffill().bfill()
                df['RSI'] = pd.Series(rsi, index=df.index).ffill().bfill()
                df['MACD'] = pd.Series(macd, index=df.index).ffill().bfill()
                df['Signal'] = pd.Series(signal, index=df.index).ffill().bfill()
                
            except Exception as e:
                logging.error(f"Error calculating indicators: {str(e)}")
                df['EMA20'] = df['Close'].rolling(20).mean()
                df['RSI'] = 50.0
                df['MACD'] = 0.0
                df['Signal'] = 0.0
            
            df['MACD_Hist'] = df['MACD'] - df['Signal']
            df['Returns'] = df['Close'].pct_change()
            df['Returns'] = df['Returns'].replace([np.inf, -np.inf], np.nan)
            
            for i in range(1, 6):
                df[f'Returns_Lag{i}'] = df['Returns'].shift(i)
                df[f'Price_Lag{i}'] = df['Close'].shift(i)
            
            df['Volatility'] = df['Returns'].rolling(window=20, min_periods=1).std()
            df['MA20'] = df['Close'].rolling(window=20, min_periods=1).mean()
            df['MA_Crossover'] = df['Close'] - df['MA20']
            df['RSI_Change'] = df['RSI'].diff()
            
            df['Target'] = np.where(df['Close'].shift(-1) > df['Close'], 1, -1)
            df = df.replace([np.inf, -np.inf], np.nan)
            df = df.ffill().bfill().dropna()
            
            logging.info(f"Prepared features shape: {df.shape}")
            return df
            
        except Exception as e:
            logging.error(f"Error in prepare_ml_features: {str(e)}")
            return pd.DataFrame()

    def split_data(self, data, train_ratio=0.8):
        """Split data with time-aware indexing"""
        dates = data.index.sort_values()
        train_size = int(len(dates) * train_ratio)
        train_dates = dates[:train_size]
        valid_dates = dates[train_size:]
        return data.loc[train_dates].copy(), data.loc[valid_dates].copy()

class MLModelHandler:
    def __init__(self):
        self.models = {
            'gradient_boosting': GradientBoostingClassifier(random_state=42),
            'random_forest': RandomForestClassifier(random_state=42),
            'logistic_regression': LogisticRegression(random_state=42)
        }
        self.regularization_results = {}
        
        self.regularization_models = {
            'lasso': Lasso(alpha=0.01, max_iter=10000),
            'ridge': Ridge(alpha=0.01, max_iter=10000),
            'elastic_net': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)
        }
        
        self.param_grids = {
            'gradient_boosting': {
                'n_estimators': [100, 200],
                'learning_rate': [0.01, 0.1],
                'max_depth': [3, 5],
                'min_samples_split': [2, 5],
                'subsample': [0.8, 1.0]
            },
            'random_forest': {
                'n_estimators': [100, 200],
                'max_depth': [3, 5],
                'min_samples_split': [2, 5],
                'max_features': ['sqrt', 'log2'],
                'bootstrap': [True],
                'ccp_alpha': [0.0, 0.1]
            },
            'logistic_regression': {
                'C': [0.1, 1.0, 10.0],
                'penalty': ['l1', 'l2', 'elasticnet'],
                'solver': ['saga'],
                'l1_ratio': [0.5]
            }
        }

    def train_models(self, X_train, y_train):
        """Train ML models with improved handling"""
        trained_models = {}
        
        try:
            X_train = np.asarray(X_train)
            y_train = np.asarray(y_train)
            
            if len(X_train.shape) == 1:
                X_train = X_train.reshape(-1, 1)
            
            if len(y_train.shape) == 2:
                y_train = y_train.ravel()
            
            param_grids = {
                'gradient_boosting': {
                    'n_estimators': [100, 200],
                    'learning_rate': [0.01, 0.05],
                    'max_depth': [3, 4],
                    'min_samples_split': [5, 10],
                    'subsample': [0.8, 0.9]
                },
                'random_forest': {
                    'n_estimators': [100, 200],
                    'max_depth': [3, 4],
                    'min_samples_split': [5, 10],
                    'min_samples_leaf': [2, 4],
                    'max_features': ['sqrt']
                },
                'logistic_regression': {
                    'C': [0.1, 1.0],
                    'class_weight': ['balanced'],
                    'max_iter': [1000],
                    'solver': ['lbfgs']
                }
            }
            
            tscv = TimeSeriesSplit(n_splits=3)
            
            for name, model in self.models.items():
                try:
                    grid_search = GridSearchCV(
                        model,
                        param_grids[name],
                        cv=tscv,
                        scoring='f1',
                        n_jobs=-1
                    )
                    
                    grid_search.fit(X_train, y_train)
                    trained_models[name] = grid_search.best_estimator_
                    
                except Exception as e:
                    logging.error(f"Error training {name} model: {str(e)}")
                    continue
                    
            return trained_models
            
        except Exception as e:
            logging.error(f"Error in train_models: {str(e)}")
            return {}

    def evaluate_models(self, X_train, y_train, X_test, y_test, models):
        """Evaluate ML models with ROC AUC"""
        results = {}
        for name, model in models.items():
            try:
                if hasattr(model, "predict_proba"):
                    y_train_proba = model.predict_proba(X_train)[:, 1]
                    y_test_proba = model.predict_proba(X_test)[:, 1]
                    train_roc_auc = roc_auc_score(y_train, y_train_proba)
                    valid_roc_auc = roc_auc_score(y_test, y_test_proba)
                else:
                    train_roc_auc = None
                    valid_roc_auc = None

                y_train_pred = model.predict(X_train)
                train_report = classification_report(y_train, y_train_pred, output_dict=True)
                
                y_test_pred = model.predict(X_test)
                test_report = classification_report(y_test, y_test_pred, output_dict=True)
                
                results[name] = {
                    'train': {
                        'roc_auc': train_roc_auc,
                        'accuracy': accuracy_score(y_train, y_train_pred),
                        'precision': precision_score(y_train, y_train_pred),
                        'recall': recall_score(y_train, y_train_pred),
                        'f1': f1_score(y_train, y_train_pred),
                        'report': train_report
                    },
                    'valid': {
                        'roc_auc': valid_roc_auc,
                        'accuracy': accuracy_score(y_test, y_test_pred),
                        'precision': precision_score(y_test, y_test_pred),
                        'recall': recall_score(y_test, y_test_pred),
                        'f1': f1_score(y_test, y_test_pred),
                        'report': test_report
                    }
                }
            except Exception as e:
                logging.error(f"Error evaluating {name} model: {str(e)}")
        return results

class Strategy1:
    def __init__(self, ema_period=20, rsi_period=14, macd_fast=12, macd_slow=26, macd_signal=9):
        self.ema_period = int(ema_period)
        self.rsi_period = int(rsi_period)
        self.macd_fast = int(macd_fast)
        self.macd_slow = int(macd_slow)
        self.macd_signal = int(macd_signal)
        self.ml_models = None
        self.best_params = {}
        self.feature_importances = {}
        self.model_metrics = {}

    def generate_signals(self, data):
        signals = pd.Series(0, index=data.index, dtype=int)
        try:
            df = data.copy()
            
            required_columns = ['Close', 'EMA20', 'RSI', 'MACD', 'Signal']
            for col in required_columns:
                if col not in df.columns:
                    df[col] = df['Close'].rolling(20).mean() if col == 'EMA20' else 0.0
                    
            long_mask = (df['Close'] > df['EMA20']) & (df['RSI'] < 65) & (df['MACD'] > df['Signal'])
            short_mask = (df['Close'] < df['EMA20']) & (df['RSI'] > 35) & (df['MACD'] < df['Signal'])
            
            signals.loc[long_mask] = 1
            signals.loc[short_mask] = -1
            
        except Exception as e:
            logging.error(f"Signal generation error: {str(e)}")
            
        return signals.fillna(0).astype(int)
    
    def optimize_ml_strategy(self, X_train, y_train):
        """Optimize ML models with regularization and visualization"""
        for model_name in ['gradient_boosting', 'random_forest']:
            if model_name == 'gradient_boosting':
                param_grid = {
                    'n_estimators': [100, 200],
                    'learning_rate': [0.01, 0.1],
                    'max_depth': [3, 5],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2],
                    'max_leaf_nodes': [10, 20],
                    'ccp_alpha': [0.0, 0.1]
                }
            else:
                param_grid = {
                    'n_estimators': [100, 200],
                    'max_depth': [3, 5],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2],
                    'max_leaf_nodes': [10, 20],
                    'max_features': ['sqrt', 'log2'],
                    'bootstrap': [True],
                    'ccp_alpha': [0.0, 0.1]
                }

            grid_search = GridSearchCV(
                self.ml_models[model_name],
                param_grid,
                cv=TimeSeriesSplit(n_splits=5),
                scoring='f1',
                n_jobs=-1
            )
            
            grid_search.fit(X_train, y_train)
            self.best_params[model_name] = grid_search.best_params_
            self.ml_models[model_name] = grid_search.best_estimator_
            
            if hasattr(self.ml_models[model_name], 'feature_importances_'):
                self.feature_importances[model_name] = self.ml_models[model_name].feature_importances_

    def optimize_parameters(self, data):
        """Оптимизация параметров стратегии с использованием TimeSeriesSplit."""
        param_grid = {
            'ema_period': range(10, 31, 5),
            'rsi_period': range(10, 21, 2),
            'macd_fast': range(8, 16, 2),
            'macd_slow': range(20, 32, 2),
            'macd_signal': range(7, 12, 1)
        }
        
        best_sharpe = -np.inf
        best_params = {}
        
        tscv = TimeSeriesSplit(n_splits=5)
        
        for params in tqdm(product(*param_grid.values()), 
                          desc="Optimizing Strategy Parameters",
                          total=np.prod([len(v) for v in param_grid.values()])):
            
            param_dict = dict(zip(param_grid.keys(), params))
            strategy = Strategy1(**param_dict)
            
            fold_sharpes = []
            for train_idx, val_idx in tscv.split(data):
                train_data = data.iloc[train_idx]
                val_data = data.iloc[val_idx]
                
                signals = strategy.generate_signals(val_data)
                returns = pd.Series(signals.shift(1) * val_data['Close'].pct_change(), 
                                  index=val_data.index)
                
                if len(returns) > 0:
                    sharpe = np.sqrt(252) * returns.mean() / returns.std() if returns.std() != 0 else 0
                    fold_sharpes.append(sharpe)
            
            avg_sharpe = np.mean(fold_sharpes) if fold_sharpes else -np.inf
            
            if avg_sharpe > best_sharpe:
                best_sharpe = avg_sharpe
                best_params = param_dict
        
        return best_params, best_sharpe

class BacktestEngine:
    def __init__(self, data_handler, model_handler):
        self.data_handler = data_handler
        self.model_handler = model_handler

    def run_backtest(self, strategy, data, train_ratio=0.8):
        try:
            if data.empty:
                raise ValueError("Empty dataset provided")
            
            train_data, valid_data = self.data_handler.split_data(data, train_ratio)
            result_data = valid_data.copy()
            
            tech_results = self.run_backtest_without_ml(strategy, data, split_data=True, train_ratio=train_ratio)
            
            features_train = self.data_handler.prepare_ml_features(train_data)
            features_valid = self.data_handler.prepare_ml_features(valid_data)
            
            feature_columns = [
                'EMA20', 'RSI', 'MACD', 'Signal', 'Returns', 'Volatility',
                'MA_Crossover', 'RSI_Change', 'MACD_Hist'
            ] + [f'Returns_Lag{i}' for i in range(1, 6)] + [f'Price_Lag{i}' for i in range(1, 6)]
            
            X_train = features_train[feature_columns].values
            y_train = features_train['Target'].values
            
            X_valid = features_valid[feature_columns].values
            y_valid = features_valid['Target'].values
            
            trained_models = self.model_handler.train_models(X_train, y_train)
            strategy.ml_models = trained_models
                        
            model_results = {}
            for name, model in trained_models.items():
                try:
                    train_pred = model.predict(X_train)
                    train_signals = pd.Series(train_pred, index=features_train.index)
                    train_returns = self._calculate_returns(train_signals, train_data)
                    train_metrics = self._calculate_metrics(train_returns)
                    
                    valid_pred = model.predict(X_valid)
                    valid_signals = pd.Series(valid_pred, index=features_valid.index)
                    valid_returns = self._calculate_returns(valid_signals, valid_data)
                    valid_metrics = self._calculate_metrics(valid_returns)
                    
                    model_results[name] = {
                        'train_metrics': train_metrics,
                        'valid_metrics': valid_metrics,
                        'signals': valid_signals,
                        'returns': valid_returns
                    }
                except Exception as e:
                    logging.error(f"Error processing {name}: {str(e)}")
                    continue
            
            return {
                'signals': tech_results['signals'],
                'returns': tech_results['returns'],
                'ml_evaluation': self.model_handler.evaluate_models(X_train, y_train, X_valid, y_valid, trained_models),
                'data': result_data,
                'model_results': model_results,
                'without_ml_metrics': {
                    'train': tech_results['train_metrics'],
                    'valid': tech_results['valid_metrics']
                }
            }
            
        except Exception as e:
            logging.error(f"Error in run_backtest: {str(e)}")
            raise
            
    def _calculate_returns(self, signals, data):
        """Safe return calculation with index alignment"""
        try:
            signals = signals.reindex(data.index, fill_value=0)
            aligned_signals = signals.ffill().bfill().fillna(0)
            position_changes = signals.diff().ffill().fillna(0)
            transaction_costs = abs(position_changes) * 0.001
            price_returns = data['Close'].pct_change()
            
            strategy_returns = pd.Series(index=data.index, dtype=float)
            mask = (signals.shift(1) != 0) & (price_returns != 0)
            strategy_returns[mask] = signals.shift(1)[mask] * price_returns[mask] - transaction_costs[mask]
            strategy_returns = strategy_returns.fillna(0)
            
            return strategy_returns
            
        except Exception as e:
            logging.error(f"Error in _calculate_returns: {str(e)}")
            return pd.Series(0, index=data.index)

    def _calculate_metrics(self, returns):
        """Calculate performance metrics with proper handling"""
        try:
            returns_series = returns if isinstance(returns, pd.Series) else pd.Series(returns)
            
            if len(returns_series) == 0:
                return {}
            
            positive_returns = returns_series[returns_series > 0]
            negative_returns = returns_series[returns_series < 0]
            
            cum_returns = (1 + returns_series).cumprod()
            rolling_max = cum_returns.expanding().max()
            drawdown = (cum_returns - rolling_max) / rolling_max
            max_drawdown = drawdown.min()
            
            trading_days = 252
            if len(returns_series) > 0 and cum_returns.iloc[-1] > 0:
                total_return = cum_returns.iloc[-1] - 1
                annualized_return = (1 + total_return) ** (trading_days/len(returns_series)) - 1
            else:
                total_return = -1
                annualized_return = -1
            
            returns_std = returns_series.std()
            negative_returns_std = negative_returns.std()
            
            sharpe_ratio = np.sqrt(trading_days) * returns_series.mean() / returns_std if returns_std != 0 else 0
            sortino_ratio = np.sqrt(trading_days) * returns_series.mean() / negative_returns_std if negative_returns_std != 0 else 0
            
            metrics = {
                'total_return': total_return,
                'annualized_return': annualized_return,
                'sharpe_ratio': sharpe_ratio,
                'sortino_ratio': sortino_ratio,
                'max_drawdown': max_drawdown,
                'win_rate': len(positive_returns) / len(returns_series) if len(returns_series) > 0 else 0,
                'profit_factor': abs(positive_returns.sum() / negative_returns.sum()) if len(negative_returns) > 0 and negative_returns.sum() != 0 else np.inf
            }
            return metrics
            
        except Exception as e:
            logging.error(f"Error in _calculate_metrics: {str(e)}")
            return {}

    def run_backtest_without_ml(self, strategy, data, split_data=True, train_ratio=0.8):
        """Run backtest without ML models with explicit train/valid split"""
        try:
            if data.empty:
                raise ValueError("Empty dataset provided")
    
            if split_data:
                train_data, valid_data = self.data_handler.split_data(data, train_ratio)
                
                # Применяем prepare_ml_features для добавления технических индикаторов
                train_data_with_features = self.data_handler.prepare_ml_features(train_data)
                valid_data_with_features = self.data_handler.prepare_ml_features(valid_data)
                
                train_signals = strategy.generate_signals(train_data_with_features)
                train_returns = self._calculate_returns(train_signals, train_data)
                train_metrics = self._calculate_metrics(train_returns)
                
                valid_signals = strategy.generate_signals(valid_data_with_features)
                valid_returns = self._calculate_returns(valid_signals, valid_data)
                valid_metrics = self._calculate_metrics(valid_returns)
                
                return {
                    'train_metrics': train_metrics,
                    'valid_metrics': valid_metrics,
                    'signals': valid_signals,
                    'returns': valid_returns
                }
            else:
                # Применяем prepare_ml_features для добавления технических индикаторов
                data_with_features = self.data_handler.prepare_ml_features(data)
                
                signals = strategy.generate_signals(data_with_features)
                returns = self._calculate_returns(signals, data)
                metrics = self._calculate_metrics(returns)
                
                return {
                    'metrics': metrics,
                    'signals': signals,
                    'returns': returns
                }
                
        except Exception as e:
            logging.error(f"Error in run_backtest_without_ml: {str(e)}")
            raise

class Dashboard:
    def __init__(self, results):
        self.results = results
         
    def _create_figure_grid(self, result, symbol):
        """Create enhanced grid of subplots with individual graphs and legends"""
        fig = make_subplots(
            rows=2, 
            cols=4,
            subplot_titles=(
                f"{symbol} Without ML", 
                f"{symbol} Gradient Boosting",
                f"{symbol} Random Forest",
                f"{symbol} Logistic Regression",
                "Cumulative Returns",
                "Win Rate",
                "Maximum Drawdown",
                "Daily Returns Distribution"
            ),
            vertical_spacing=0.15,
            horizontal_spacing=0.05,
            specs=[[{}, {}, {}, {}], [{}, {}, {}, {}]]
        )
        
        strategies = {
            'without_ml': {'name': 'Without ML', 'color': 'gray'},
            'gradient_boosting': {'name': 'Gradient Boosting', 'color': 'green'},
            'random_forest': {'name': 'Random Forest', 'color': 'blue'},
            'logistic_regression': {'name': 'Logistic Regression', 'color': 'red'}
        }
        
        prices = result['with_ml']['data']['Close']
        
        for col, (strategy_key, props) in enumerate(strategies.items(), 1):
            try:
                if strategy_key == 'without_ml':
                    signals = result['without_ml']['signals']
                    returns = result['without_ml']['returns']
                else:
                    model_data = result['with_ml']['model_results'].get(strategy_key)
                    signals = model_data['signals'].reindex(prices.index, fill_value=0)
                    returns = model_data['returns'].reindex(prices.index, fill_value=0)
                
                fig.add_trace(
                    go.Scatter(
                        x=prices.index,
                        y=prices,
                        name="Price",
                        line=dict(color='black', width=1),
                        legendgroup=f'group{col}',
                        showlegend=False
                    ),
                    row=1, col=col
                )
                
                buy_signals = signals[signals > 0]
                fig.add_trace(
                    go.Scatter(
                        x=buy_signals.index,
                        y=prices[buy_signals.index],
                        mode='markers',
                        name='Buy',
                        marker=dict(symbol='triangle-up', color='green', size=8),
                        legendgroup=f'group{col}',
                        showlegend=True
                    ),
                    row=1, col=col
                )
                
                sell_signals = signals[signals < 0]
                fig.add_trace(
                    go.Scatter(
                        x=sell_signals.index,
                        y=prices[sell_signals.index],
                        mode='markers',
                        name='Sell',
                        marker=dict(symbol='triangle-down', color='red', size=8),
                        legendgroup=f'group{col}',
                        showlegend=True
                    ),
                    row=1, col=col
                )
                
                cum_returns = (1 + returns).cumprod() - 1
                win_rate = (returns > 0).rolling(window=30).mean()
                rolling_max = cum_returns.expanding().max()
                drawdown = (cum_returns - rolling_max) / rolling_max
                
                fig.add_trace(
                    go.Scatter(
                        x=cum_returns.index,
                        y=cum_returns * 100,
                        name=props['name'],
                        line=dict(color=props['color']),
                        showlegend=True
                    ),
                    row=2, col=1
                )
                
                fig.add_trace(
                    go.Scatter(
                        x=win_rate.index,
                        y=win_rate * 100,
                        name=props['name'],
                        line=dict(color=props['color']),
                        showlegend=False
                    ),
                    row=2, col=2
                )
                
                fig.add_trace(
                    go.Scatter(
                        x=drawdown.index,
                        y=drawdown * 100,
                        name=props['name'],
                        line=dict(color=props['color']),
                        showlegend=False
                    ),
                    row=2, col=3
                )
                
                fig.add_trace(
                    go.Histogram(
                        x=returns * 100,
                        name=props['name'],
                        marker_color=props['color'],
                        opacity=0.7,
                        showlegend=False
                    ),
                    row=2, col=4
                )
                
            except Exception as e:
                logging.error(f"Error processing {strategy_key}: {str(e)}")
                continue
        
        fig.update_layout(
            height=1000,
            width=1600,
            showlegend=True,
            margin=dict(t=100, b=50, l=50, r=50),
            template="plotly_white"
        )
        
        for i in range(1, 5):
            fig.update_yaxes(title_text="Price", row=1, col=i)
            fig.update_xaxes(title_text="Date", row=1, col=i)
            
        fig.update_yaxes(title_text="Return (%)", row=2, col=1)
        fig.update_yaxes(title_text="Win Rate (%)", row=2, col=2)
        fig.update_yaxes(title_text="Drawdown (%)", row=2, col=3)
        fig.update_yaxes(title_text="Count", row=2, col=4)
        
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_xaxes(title_text="Date", row=2, col=2)
        fig.update_xaxes(title_text="Date", row=2, col=3)
        fig.update_xaxes(title_text="Daily Return (%)", row=2, col=4)
        
        return fig
       
    def create_metrics_table(self, result):
        """Create a metrics comparison table with train/valid split including Without ML metrics"""
        metrics_data = {
            'Metric': [
                'Total Return (%)',
                'Annualized Return (%)',
                'Sharpe Ratio',
                'Sortino Ratio',
                'Max Drawdown (%)',
                'Win Rate (%)',
                'Profit Factor'
            ]
        }
        
        if 'without_ml_metrics' in result:
            without_ml_train = result['without_ml_metrics']['train']
            metrics_data['Without ML (Train)'] = [
                f"{without_ml_train.get('total_return', 0)*100:.2f}",
                f"{without_ml_train.get('annualized_return', 0)*100:.2f}",
                f"{without_ml_train.get('sharpe_ratio', 0):.2f}",
                f"{without_ml_train.get('sortino_ratio', 0):.2f}",
                f"{without_ml_train.get('max_drawdown', 0)*100:.2f}",
                f"{without_ml_train.get('win_rate', 0)*100:.2f}",
                f"{without_ml_train.get('profit_factor', 0):.2f}"
            ]
            
            without_ml_valid = result['without_ml_metrics']['valid']
            metrics_data['Without ML (Valid)'] = [
                f"{without_ml_valid.get('total_return', 0)*100:.2f}",
                f"{without_ml_valid.get('annualized_return', 0)*100:.2f}",
                f"{without_ml_valid.get('sharpe_ratio', 0):.2f}",
                f"{without_ml_valid.get('sortino_ratio', 0):.2f}",
                f"{without_ml_valid.get('max_drawdown', 0)*100:.2f}",
                f"{without_ml_valid.get('win_rate', 0)*100:.2f}",
                f"{without_ml_valid.get('profit_factor', 0):.2f}"
            ]
        
        models = [
            ('gradient_boosting', 'Gradient Boosting'),
            ('random_forest', 'Random Forest'), 
            ('logistic_regression', 'Logistic Regression')
        ]
        
        for model_key, model_name in models:
            if model_key in result['with_ml']['model_results']:
                for prefix, source in [('Train', 'train_metrics'), ('Valid', 'valid_metrics')]:
                    metrics = result['with_ml']['model_results'][model_key][source]
                    col_name = f"{model_name} ({prefix})"
                    
                    metrics_data[col_name] = [
                        f"{metrics.get('total_return', 0) * 100:.2f}",
                        f"{metrics.get('annualized_return', 0) * 100:.2f}",
                        f"{metrics.get('sharpe_ratio', 0):.2f}",
                        f"{metrics.get('sortino_ratio', 0):.2f}",
                        f"{metrics.get('max_drawdown', 0) * 100:.2f}",
                        f"{metrics.get('win_rate', 0) * 100:.2f}",
                        f"{metrics.get('profit_factor', 0):.2f}"
                    ]
        
        df = pd.DataFrame(metrics_data)
        
        ordered_cols = ['Metric']
        if 'Without ML (Train)' in df.columns:
            ordered_cols.extend(['Without ML (Train)', 'Without ML (Valid)'])
        
        for model_name in ['Gradient Boosting', 'Random Forest', 'Logistic Regression']:
            train_col = f"{model_name} (Train)"
            valid_col = f"{model_name} (Valid)"
            if train_col in df.columns:
                ordered_cols.extend([train_col, valid_col])
        
        return df[ordered_cols].set_index('Metric')
        
    def create_ml_metrics_table(self, result):
        """Create ML metrics table with ROC AUC"""
        ml_metrics = {
            'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC']
        }
        
        if 'with_ml' in result and 'ml_evaluation' in result['with_ml']:
            for model_name, metrics in result['with_ml']['ml_evaluation'].items():
                train_metrics = [
                    f"{metrics['train']['accuracy']:.4f}",
                    f"{metrics['train']['precision']:.4f}",
                    f"{metrics['train']['recall']:.4f}",
                    f"{metrics['train']['f1']:.4f}",
                    f"{metrics['train']['roc_auc']:.4f}" if metrics['train']['roc_auc'] is not None else 'N/A'
                ]
                
                valid_metrics = [
                    f"{metrics['valid']['accuracy']:.4f}",
                    f"{metrics['valid']['precision']:.4f}",
                    f"{metrics['valid']['recall']:.4f}",
                    f"{metrics['valid']['f1']:.4f}",
                    f"{metrics['valid']['roc_auc']:.4f}" if metrics['valid']['roc_auc'] is not None else 'N/A'
                ]
                
                ml_metrics[f'{model_name} (Train)'] = train_metrics
                ml_metrics[f'{model_name} (Valid)'] = valid_metrics
        
        ordered_columns = ['Metric']
        model_order = ['gradient_boosting', 'random_forest', 'logistic_regression']
        for model in model_order:
            if f'{model} (Train)' in ml_metrics:
                ordered_columns.extend([f'{model} (Train)', f'{model} (Valid)'])
        
        return pd.DataFrame(ml_metrics)[ordered_columns].set_index('Metric')

    def show_notebook(self):
        """Display the dashboard in Jupyter Notebook"""
        display(HTML("<h1 style='text-align: center;'>Trading Strategy Analysis Dashboard</h1>"))
        
        if not self.results:
            display(HTML("<h3>No data available to display</h3>"))
            return
        
        symbols = list(self.results.keys())
        dropdown = widgets.Dropdown(
            options=symbols,
            description='Symbol:',
            style={'description_width': 'initial'}
        )
        
        def update_dashboard(symbol):
            if symbol in self.results:
                result = self.results[symbol]
                
                display(HTML("<h3>Performance Metrics</h3>"))
                metrics_df = self.create_metrics_table(result)
                display(metrics_df.style.background_gradient(cmap='RdYlGn', axis=1))
                
                display(HTML("<h3>ML Model Metrics</h3>"))
                ml_metrics_df = self.create_ml_metrics_table(result)
                display(ml_metrics_df.style.background_gradient(cmap='RdYlGn', axis=1))
                
                fig = self._create_figure_grid(result, symbol)
                fig.show()
        
        widgets.interact(update_dashboard, symbol=dropdown)

def main():
    stocks = ['GBTC', 'BITF']
    cryptos = ['BTC', 'ETH']
    symbols = stocks + cryptos
    start_date = '2019-10-30'

    try:
        data_handler = DataHandler()
        model_handler = MLModelHandler()
        backtest_engine = BacktestEngine(data_handler, model_handler)

        results = {}
        
        for symbol in symbols:
            try:
                logging.info(f"\n{'='*50}\nProcessing {symbol}\n{'='*50}")
                
                data_dict = data_handler.load_data([symbol], start_date)
                data = data_dict.get(symbol)

                if data is None or data.empty:
                    logging.error(f"Skipping {symbol} - no valid data available")
                    continue

                if data.shape[0] < 100:
                    logging.error(f"Skipping {symbol} - insufficient data points ({data.shape[0]})")
                    continue

                with tqdm(total=3, desc=f"Processing {symbol}") as pbar:
                    processed_data = data_handler.preprocess_data(data)
                    pbar.update(1)
                    
                    normalized_data = data_handler.normalize_data(processed_data)
                    pbar.update(1)
                    
                    if normalized_data.empty:
                        logging.error(f"Skipping {symbol} - empty data after normalization")
                        continue
                        
                    strategy = Strategy1()
                    pbar.update(1)

                try:
                    # Передаем normalized_data для ML-моделей и processed_data для стратегии без ML
                    result_with_ml = backtest_engine.run_backtest(strategy, normalized_data)
                    result_without_ml = backtest_engine.run_backtest_without_ml(strategy, processed_data)
                    
                    results[symbol] = {
                        'with_ml': result_with_ml,
                        'without_ml': result_without_ml
                    }
                    logging.info(f"Successfully completed processing for {symbol}")
                    
                except Exception as e:
                    logging.error(f"Backtesting failed for {symbol}: {str(e)}")
                    continue
                    
            except Exception as e:
                logging.error(f"Fatal error processing {symbol}: {str(e)}")
                continue

        if results:
            dashboard = Dashboard(results)
            dashboard.show_notebook()
        else:
            logging.error("No results to display - all symbols failed")
            
    except Exception as e:
        logging.error(f"Fatal error in main: {str(e)}")
        raise

if __name__ == "__main__":
    main()

2025-04-21 22:40:13,481 - INFO - 
Processing GBTC
2025-04-21 22:40:13,482 - INFO - Processing data for GBTC
2025-04-21 22:40:13,492 - INFO - Loaded data for GBTC from local file: C:\Users\MV\DataSC_raw\GBTC.csv
2025-04-21 22:40:13,495 - INFO - Successfully loaded local data for GBTC: 1374 records
Preprocessing Data: 100%|██████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 1250.31it/s]
2025-04-21 22:40:13,509 - INFO - Successfully preprocessed data: 1374 records

Processing GBTC: 100%|██████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 103.41it/s]
2025-04-21 22:40:13,558 - INFO - Prepared features shape: (1099, 22)
2025-04-21 22:40:13,587 - INFO - Prepared features shape: (275, 22)
2025-04-21 22:40:13,636 - INFO - Prepared features shape: (1099, 22)
2025-04-21 22:40:13,660 - INFO - Prepared features shape: (275, 22)
2025-04-21 22:41:03,808 - INFO - Prepared features shape: (1099, 22)
2025-04-21 22:41:03,835 - INFO - Prepare

interactive(children=(Dropdown(description='Symbol:', options=('GBTC', 'BITF', 'BTC', 'ETH'), style=Descriptio…